# Homework 3 — Lead Scoring Classifier (ML Zoomcamp 2026)

**Dataset:** `course_lead_scoring_2026.csv` — pinned 2026 release (`cohorts/2026/data/`). Local copy first, else official homework URL.

**Goal:** predict `converted` with logistic regression. Workflow follows the [03-classification module](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/main/03-classification).

Questions **Q1–Q6** + Answers Summary + self-verification cell.

Form: <https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw03>

## Setup

Imports. (Comentarios en español dentro del código.)

In [1]:
# Importamos pandas/numpy para datos y sklearn para el modelo
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

## Load the dataset

In [2]:
# Cargamos el CSV local (copia fijada); si no existe, URL oficial
DATA_URL = ('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/'
            'cohorts/2026/data/course_lead_scoring_2026.csv')
local = Path('course_lead_scoring_2026.csv')
src = str(local) if local.exists() else DATA_URL
print('Source:', src)
df = pd.read_csv(src)
print('Shape:', df.shape)
df.head()

Source: course_lead_scoring_2026.csv
Shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

Missing: categoricals -> `'NA'`, numericals -> `0.0`.

In [3]:
# Revisamos faltantes; rellenamos según el enunciado
print(df.isnull().sum())
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]):
        df[col] = df[col].fillna('NA')  # categorica: marcador explicito
    else:
        df[col] = df[col].fillna(0.0)  # numerica: cero neutro
print('\nMissing after fill:', int(df.isnull().sum().sum()))

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

Missing after fill: 0


## Q1 — Most frequent `industry` (mode)

Options: `NA`, `technology`, `healthcare`, `retail`.

In [4]:
# Q1: la moda = valor mas frecuente de 'industry'
mode_industry = df['industry'].mode()[0]
print(df['industry'].value_counts().head())
print(f'\nRespuesta Q1: {mode_industry}')

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64

Respuesta Q1: technology


## Q2 — Correlation matrix (numerical features)

Pearson correlation; check the four listed pairs only.

In [5]:
# Q2: correlacion de Pearson por pares (features numericas)
num_cols = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
corr = df[num_cols].corr()
print(corr.round(4))
pairs = [('interaction_count', 'lead_score'), ('number_of_courses_viewed', 'lead_score'),
         ('number_of_courses_viewed', 'interaction_count'), ('annual_income', 'interaction_count')]
for a, b in pairs:
    print(f'{a} x {b}: {corr.loc[a, b]:.4f}')
winner = max(pairs, key=lambda p: corr.loc[p[0], p[1]])
print(f'\nRespuesta Q2: {winner[0]} and {winner[1]}')

                          annual_income  number_of_courses_viewed  \
annual_income                    1.0000                    0.1613   
number_of_courses_viewed         0.1613                    1.0000   
interaction_count                0.1228                    0.7216   
lead_score                       0.2295                    0.7572   

                          interaction_count  lead_score  
annual_income                        0.1228      0.2295  
number_of_courses_viewed             0.7216      0.7572  
interaction_count                    1.0000      0.9157  
lead_score                           0.9157      1.0000  


interaction_count x lead_score: 0.9157
number_of_courses_viewed x lead_score: 0.7572
number_of_courses_viewed x interaction_count: 0.7216
annual_income x interaction_count: 0.1228

Respuesta Q2: interaction_count and lead_score


## Split the data

Exact homework calls (`random_state=42`): 3000 train / 1000 val / 1000 test.

In [6]:
# Split exacto del enunciado; separamos el target 'converted'
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
print(len(df_train), len(df_val), len(df_test))
y_train = df_train['converted'].values
y_val = df_val['converted'].values

3000 1000 1000


## Q3 — Mutual information (train only)

Higher MI = more informative about `converted`. Rounded to 2 decimals.

In [7]:
# Q3: info mutua vs 'converted' (solo train); columna a codigos enteros
cat_cols = ['lead_source', 'industry', 'employment_status', 'location']
mi_scores = {}
for col in cat_cols:
    Xc = df_train[[col]].apply(lambda s: s.astype('category').cat.codes)
    score = mutual_info_classif(Xc, y_train, discrete_features=True, random_state=42)[0]
    mi_scores[col] = round(float(score), 2)
    print(f'{col}: {mi_scores[col]}')
print(f'\nRespuesta Q3: {max(mi_scores, key=mi_scores.get)}')

lead_source: 0.03
industry: 0.0
employment_status: 0.02
location: 0.0

Respuesta Q3: lead_source


## Q4 — Logistic regression + one-hot encoding

Fit on train, accuracy on validation (2 decimals).

In [8]:
# Q4: get_dummies = one-hot; alineamos columnas de val con train
features = ['lead_source', 'industry', 'employment_status', 'location',
            'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
def make_X(dfin, cols):
    return pd.get_dummies(dfin[cols], drop_first=False, dtype=float)
X_train = make_X(df_train, features)
X_val = make_X(df_val, features).reindex(columns=X_train.columns, fill_value=0.0)
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
acc = accuracy_score(y_val, model.predict(X_val))
print(f'accuracy: {acc:.4f} -> round 2: {round(acc, 2)}')
print(f'\nRespuesta Q4: {round(acc, 2)}')

accuracy: 0.6450 -> round 2: 0.65

Respuesta Q4: 0.65


## Q5 — Feature elimination

Retrain without one feature at a time; smallest `base - without` = least useful.

In [9]:
# Q5: quitamos una feature original cada vez y medimos la diferencia
base_acc = acc
diffs = {}
for feat in features:
    keep = [f for f in features if f != feat]
    Xt = make_X(df_train, keep)
    Xv = make_X(df_val, keep).reindex(columns=Xt.columns, fill_value=0.0)
    mm = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    mm.fit(Xt, y_train)
    aa = accuracy_score(y_val, mm.predict(Xv))
    diffs[feat] = base_acc - aa
    print(f'sin {feat}: diff={diffs[feat]:+.4f}')
subset = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
best = min(subset, key=lambda k: abs(diffs[k]))
print(f'\nRespuesta Q5: {best}')

sin lead_source: diff=+0.0030


sin industry: diff=+0.0000
sin employment_status: diff=+0.0000
sin location: diff=+0.0000


sin annual_income: diff=-0.0790


sin number_of_courses_viewed: diff=+0.0020


sin interaction_count: diff=+0.0440
sin lead_score: diff=+0.0010

Respuesta Q5: number_of_courses_viewed


## Q6 — Regularized logistic regression

Smaller `C` = stronger regularization. Best val accuracy wins; ties -> smallest `C`.

In [10]:
# Q6: probamos C pequenos (mas regularizacion) a grandes
q6_scores = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    mm = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    mm.fit(X_train, y_train)
    aa = accuracy_score(y_val, mm.predict(X_val))
    q6_scores[C] = round(float(aa), 3)
    print(f'C={C}: acc={aa:.4f} -> round 3: {q6_scores[C]}')
# mejor = mayor accuracy; en empate, el C mas pequeno
best_acc = max(q6_scores.values())
best_C = min([c for c, v in q6_scores.items() if v == best_acc])
print(f'\nRespuesta Q6: {best_C}')

C=1e-06: acc=0.5980 -> round 3: 0.598
C=1e-05: acc=0.5980 -> round 3: 0.598


C=0.0001: acc=0.6130 -> round 3: 0.613
C=0.001: acc=0.6450 -> round 3: 0.645

Respuesta Q6: 0.001


## Answers Summary (copy into the submission form)

| Question | Answer |
|---|---|
| Q1 — mode of `industry` | `technology` |
| Q2 — biggest correlation | `interaction_count and lead_score` (0.9157) |
| Q3 — biggest mutual information | `lead_source` (0.03) |
| Q4 — logistic regression accuracy | `0.65` |
| Q5 — least useful feature | `number_of_courses_viewed` |
| Q6 — best `C` | `0.001` |


## Self-verification

Asserts every answer against the computed value and the official options.

In [11]:
# Autocomprobacion: cada respuesta vs. lo calculado y las opciones oficiales
assert mode_industry in ('NA', 'technology', 'healthcare', 'retail'), mode_industry
assert mode_industry == 'technology', mode_industry
assert set(winner) == {'interaction_count', 'lead_score'}, winner
assert max(mi_scores, key=mi_scores.get) == 'lead_source', mi_scores
assert round(acc, 2) == 0.65 and round(acc, 2) in (0.55, 0.65, 0.75, 0.85)
assert best == 'number_of_courses_viewed', diffs
assert best_C == 0.001 and q6_scores == {0.000001: 0.598, 0.00001: 0.598, 0.0001: 0.613, 0.001: 0.645}
print('OK: todas las respuestas verificadas')

OK: todas las respuestas verificadas
